In [1]:
import pandas as pd
import numpy as np
from pandas.core.interchange.dataframe_protocol import DataFrame
from sklearn.preprocessing import StandardScaler

In [26]:
#reading the  data
reviews_df = pd.read_csv('data/reviews.csv')
listings_df = pd.read_csv('data/listings.csv')

In [27]:
# method to fill NaN values with the mean of the column
def fill_na_with_mean(df, column):
    if column in df.columns:
        mean_value = df[column].mean()
        df[column].fillna(mean_value, inplace=True)
    else:
        raise ValueError(f"Column {column} does not exist in the DataFrame.")

In [28]:
def clean_listings_df():
    # remove columns with too many missing values or unnecessary information
    listings_df.drop(columns=[
        'neighborhood_overview', 'host_about', 'host_location',
        'host_response_time', 'host_response_rate', 'host_acceptance_rate',
        'host_is_superhost', 'host_neighbourhood', 'host_verifications',
        'host_thumbnail_url', 'license', 'calendar_updated',
        'calendar_last_scraped', 'last_review', 'first_review', 'neighbourhood_group_cleansed',
        'last_scraped', 'source', 'neighbourhood', 'calculated_host_listings_count', 'calculated_host_listings_count_entire_homes',
        'calculated_host_listings_count_private_rooms', 'calculated_host_listings_count_shared_rooms',
    ], inplace=True)
    # remove rows with missing values in important columns
    listings_df.dropna(subset=['price', 'latitude', 'longitude', 'accommodates', 'bedrooms', 'beds', 'has_availability'], inplace=True)
    listings_df['description'].fillna('no description', inplace=True)

    # fill NaN values in review score columns with the mean of the column
    for col in ['review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
                'review_scores_checkin', 'review_scores_communication',
                'review_scores_location', 'review_scores_value', 'reviews_per_month']:
        # fill NaN values with the mean of the column
        fill_na_with_mean(listings_df, col)


    # turn price column into float, removing dollar sign
    listings_df['price'] = listings_df['price'].str.replace('[$,]', '', regex=True).astype(float)
    # change id column name to listing_id for consistency with reviews_df
    listings_df.rename(columns={'id': 'listing_id'}, inplace=True)
    
    
    return listings_df

In [29]:
def clean_reviews_df():
    # remove columns with too many missing values or unnecessary information
    # TODO: Add columns to drop if necessary. I think if we use this for sentiment analysis, we dont need a lot of the columns.
    reviews_df.drop(columns=['date', 'reviewer_id', 'reviewer_name'], inplace=True)

    # remove rows with missing values in important columns, in this case no comments
    reviews_df.dropna(subset=['comments'], inplace=True)
    return reviews_df

In [30]:
def join_dfs(*dfs, join_key='listing_id', how='left'):
    if not dfs:
        raise ValueError("At least one DataFrame must be provided.")

    # Ensure join_key is set as index for optimization (optional)
    merged_df = dfs[0]
    for df in dfs[1:]:
        merged_df = pd.merge(merged_df, df, on=join_key, how=how, copy=False, sort=False)
    return merged_df

In [31]:
def export_processed_data(arg):
    if len(arg) == 0:
        raise ValueError("At least one DataFrame must be provided.")
    arg.to_csv('./processed-data/clean-data.csv', index=False)

In [32]:
nDf = join_dfs(clean_listings_df(), clean_reviews_df())
print(nDf.head())
print("finished dataframe shape ", nDf.shape)

# export_processed_data(nDf)
# TODO: the finished csv file looks kinda weird. get rid of the <br> and such?

C:\Users\fabia\AppData\Local\Temp\ipykernel_22992\1752424774.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  listings_df['description'].fillna('no description', inplace=True)
C:\Users\fabia\AppData\Local\Temp\ipykernel_22992\3691534955.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves a

   listing_id                        listing_url       scrape_id  \
0        3781  https://www.airbnb.com/rooms/3781  20250315060211   
1        3781  https://www.airbnb.com/rooms/3781  20250315060211   
2        3781  https://www.airbnb.com/rooms/3781  20250315060211   
3        3781  https://www.airbnb.com/rooms/3781  20250315060211   
4        3781  https://www.airbnb.com/rooms/3781  20250315060211   

                        name  \
0  HARBORSIDE-Walk to subway   
1  HARBORSIDE-Walk to subway   
2  HARBORSIDE-Walk to subway   
3  HARBORSIDE-Walk to subway   
4  HARBORSIDE-Walk to subway   

                                         description  \
0  Fully separate apartment in a two apartment bu...   
1  Fully separate apartment in a two apartment bu...   
2  Fully separate apartment in a two apartment bu...   
3  Fully separate apartment in a two apartment bu...   
4  Fully separate apartment in a two apartment bu...   

                                         picture_url  host_id

In [33]:
nDf.dtypes

listing_id                       int64
listing_url                     object
scrape_id                        int64
name                            object
description                     object
picture_url                     object
host_id                          int64
host_url                        object
host_name                       object
host_since                      object
host_picture_url                object
host_listings_count            float64
host_total_listings_count      float64
host_has_profile_pic            object
host_identity_verified          object
neighbourhood_cleansed          object
latitude                       float64
longitude                      float64
property_type                   object
room_type                       object
accommodates                     int64
bathrooms                      float64
bathrooms_text                  object
bedrooms                       float64
beds                           float64
amenities                

In [34]:
#dropping all id coloumns (just for testing)
nDf = nDf.loc[:, ~nDf.columns.str.contains('id', case=False)]
todo_df = nDf.select_dtypes(include=['int', 'float64'])

In [35]:

columns_to_drop = ["minimum_minimum_nights" , "maximum_minimum_nights" , "minimum_maximum_nights" , "maximum_maximum_nights", "estimated_revenue_l365d" ]
todo_df = todo_df.drop(columns=columns_to_drop)


In [36]:
todo_df.dtypes

host_listings_count            float64
host_total_listings_count      float64
latitude                       float64
longitude                      float64
accommodates                     int64
bathrooms                      float64
bedrooms                       float64
beds                           float64
price                          float64
minimum_nights                   int64
maximum_nights                   int64
minimum_nights_avg_ntm         float64
maximum_nights_avg_ntm         float64
availability_30                  int64
availability_60                  int64
availability_90                  int64
availability_365                 int64
number_of_reviews                int64
number_of_reviews_ltm            int64
number_of_reviews_l30d           int64
availability_eoy                 int64
number_of_reviews_ly             int64
estimated_occupancy_l365d        int64
review_scores_rating           float64
review_scores_accuracy         float64
review_scores_cleanliness

In [37]:
from sklearn.ensemble import RandomForestRegressor  
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
import time 

In [38]:
correlations = todo_df.corr(numeric_only=True)['price'].sort_values(ascending=False)
print(correlations)

price                          1.000000
accommodates                   0.610908
bedrooms                       0.568140
beds                           0.510498
bathrooms                      0.424247
review_scores_location         0.206543
review_scores_cleanliness      0.132354
availability_eoy               0.130056
availability_365               0.111107
latitude                       0.108638
review_scores_rating           0.096535
longitude                      0.091535
maximum_nights                 0.066407
review_scores_communication    0.056058
number_of_reviews_l30d         0.053433
review_scores_accuracy         0.045879
host_total_listings_count      0.041085
maximum_nights_avg_ntm         0.031489
host_listings_count            0.027627
estimated_occupancy_l365d      0.023251
review_scores_checkin          0.012779
availability_90                0.011298
number_of_reviews_ltm          0.003623
number_of_reviews_ly          -0.002546
availability_60               -0.003143


In [20]:
features = todo_df.drop(columns=['price'])  
targets = todo_df['price']


In [42]:
features.dtypes

host_listings_count            float64
host_total_listings_count      float64
latitude                       float64
longitude                      float64
accommodates                     int64
bathrooms                      float64
bedrooms                       float64
beds                           float64
minimum_nights                   int64
maximum_nights                   int64
minimum_nights_avg_ntm         float64
maximum_nights_avg_ntm         float64
availability_30                  int64
availability_60                  int64
availability_90                  int64
availability_365                 int64
number_of_reviews                int64
number_of_reviews_ltm            int64
number_of_reviews_l30d           int64
availability_eoy                 int64
number_of_reviews_ly             int64
estimated_occupancy_l365d        int64
review_scores_rating           float64
review_scores_accuracy         float64
review_scores_cleanliness      float64
review_scores_checkin    

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(features, targets, test_size=0.3, random_state=1)               

In [44]:
# random forest model
print("Training model...")
for _ in tqdm(range(1), desc="Fitting Random Forest"):
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

Training model...


Fitting Random Forest: 100%|██████████| 1/1 [01:12<00:00, 72.86s/it]


In [23]:
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"✅ Training complete. MSE: {mse:.2f}")

✅ Training complete. MSE: 68.95


In [24]:
# Create a DataFrame for comparison
comparison_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred
})

# Show a few rows
print(comparison_df.head(10))

   Actual  Predicted
0   110.0      110.0
1   169.0      169.0
2   200.0      200.0
3   150.0      150.0
4   318.0      318.0
5   130.0      130.0
6   141.0      141.0
7   106.0      106.0
8   104.0      104.0
9   161.0      161.0


In [25]:
print(comparison_df[comparison_df['Actual'] != comparison_df['Predicted']].head(20))

      Actual  Predicted
26     150.0     159.25
69      64.0      58.95
99     127.0     130.82
193    274.0     262.02
322     58.0      60.18
498    424.0     423.68
557    135.0     275.32
630    279.0     275.06
747     68.0      80.56
785    205.0     193.02
881    237.0     240.24
882     80.0      69.56
972     61.0      67.34
1035    35.0      35.75
1050   338.0     319.07
1071    69.0      72.67
1292   413.0     245.77
1350    67.0     140.40
1410   133.0     132.06
1438   161.0     139.44


In [20]:
import shap

# Create a SHAP explainer for tree-based models
explainer = shap.TreeExplainer(model)

# Compute SHAP values
shap_values = explainer.shap_values(X_test)

: 

: 

In [ ]:
shap.summary_plot(shap_values, X_test)